# Activity: Variations on Linear Regression

## Learning Objectives

By the end of this activity, you will be able to:

-   Implement a variety of scoring functions for fitting linear
    regression models.
-   Describe how different scoring functions affect the fitted model,
    especially in the presence of outliers or unequal costs of errors.

## Intro: Variations on Linear Regression

As we saw in lecture, a *model* is a function $f$ that accepts an input
$x$ and produces a prediction $\hat{y} = f(x)$. In *linear regression*,
we assume that the model is a linear function of the input, which for
one input variable looks like

<span id="eq-linear-model">$$
\begin{aligned}
    \hat{y} = f(x) = wx+b\;. 
\end{aligned}
 \qquad(1)$$</span>

We can think of this equation as describing a *family* of possible
models, one for each choice of the parameters $w$ (the slope) and $b$
(the intercept). In the *training* process, we search for values of the
parameters that “fit” our data well in some way. In the notes, we used
the *mean squared error* (MSE) as our measure of fit:

$$
\begin{aligned}
    \text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2\;. 
\end{aligned}
$$

We then showed an example of selecting among a set of possible models by
choosing the one that minimized the MSE on our training data.

However, MSE is not the only way to measure how well a model fits the
data – there are many approaches! In this activity, you’ll design
different measures of fit that may reflect features in the data or
decision-making priorities.

### Data Set

Throughout this lab, we’ll work with the following synthetic data set:

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set_theme(style = "whitegrid")

def data_with_outliers(n_points = 50, n_outliers = 5, random_seed = 42):
    np.random.seed(random_seed)
    x = np.linspace(0, 10, n_points)
    y = 2 * x + 1 + np.random.normal(0, 2, n_points)

    # Add outliers
    outlier_indices = np.random.choice(n_points, n_outliers, replace=False)
    y[outlier_indices] += np.random.normal(20, 5, n_outliers)

    return pd.DataFrame({'x': x, 'y': y})

data = data_with_outliers()

sns.scatterplot(data=data, x='x', y='y')

This data set includes a relatively well-defined linear trend in
combination with several *outliers*, points which deviate considerably
from the dominant trend.

## Part A: Implement Model Fitting via Grid Search

### Exercise A1

In the code block below, please implement a function `fit_linear_model`.
This function should accept four arguments:

-   `data`: a DataFrame with columns `x` and `y` representing the input
    and output variables, respectively.
-   `score_function`: the scoring function used to evaluate the model
    fit.
    -   Assume that the `score_function` has signature
        `score_function(y_true, y_pred, **score_kwargs)`, where `y_true`
        is a vector of true output values, `y_pred` is a vector of
        predicted output values, and `score_kwargs` are any additional
        keyword arguments needed by the scoring function.
-   `n_grid`: the number of values for the slope and intercept to
    evaluate in a grid search.
-   `**score_kwargs`: any additional keyword arguments needed by the
    scoring function.

Here’s what your function should do:

1.  Generate a list of `n_grid` evenly-spaced values for the slope `w`.
    Generate similar list of `n_grid` evenly-spaced values for the
    intercept `b` over a reasonable range (e.g., from `0` to `4`).
2.  For each combination of `w` and `b` from these lists, compute the
    predicted output values for the input data using
    <a href="#eq-linear-model" class="quarto-xref">Equation 1</a>.
3.  Use the provided `score_function` to compute a score for each model
    based on the true output values and the predicted output values.
4.  Return a tuple `(w, b, s)` representing the slope, intercept, and
    score of the best-fitting model (i.e., the model with the lowest
    score).

***Note***: In the accompanying
[notes](https://middcs.github.io/data-science-notes/chapters/20-introduction-prediction.html),
we did something similar using a data frame, which you may wish to
adapt. <span class="column-margin margin-aside">In the notes, we picked
100 random pairs for $w$ and $b$ instead of systematically searching on
a grid.</span> It’s also fine to use a for-loop approach if you’d
prefer.

In [ ]:
# TODO: Your code here

## Exercise A2

Now, test your implementation. Verify that your `fit_linear_model`
function works correctly by using it to fit a model using the mean
squared error (MSE) as the scoring function. You can do this by running
the code block below:

In [ ]:
def mse (y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

w, b, score = fit_linear_model(data, mse, n_grid = 100)

Now, please write code to construct a plot of your fitted model against
the data. You can do this by computing the predicted output values using
your learned parameters `w` and `b`, and then plotting these predictions
as a line on top of a scatter plot of the data points.

In [ ]:
# TODO: Your code here

*We are expecting you to find that the best-fit regression line doesn’t
match the dominant trend very well due to the influence of outliers in
the data*.

Linear regression with the MSE is sensitive to outliers, meaning that a
few extreme data points can have a large influence on the fitted model.
In the next several sections, we’ll explore alternative scoring
functions that allow models to detect the dominant trend even in the
presence of outliers.

## Part B: Alternative Score Functions

### Exercise B1: Mean Absolute Error Regression

Rather than using the mean-*squared* error as our scoring function, we
can instead use the mean-*absolute* error (MAE):

$$
\begin{aligned}
    MAE = \frac{1}{n}\sum_{i = 1}^n \left|\hat{y}_i - y_i\right|\;.
\end{aligned}
$$

Please implement the MAE scoring function. Then,

1.  Use your `fit_linear_model` function to fit a model using MAE as the
    scoring function.
2.  Plot the resulting model predictions against the data, and compare
    to the fit you got using the MSE. Please make sure that the two
    prediction lines are shown on the same plot for easy comparison, and
    that they are distinguished using an appropriately-labeled legend.
3.  Finally, comment on your findings and the comparison between the two
    learned models.

In [ ]:
# TODO: Your code here

*[TODO: Your response here]*

*Mathematically, the reason the MAE is a little less sensitive to
outliers is that the contribution of a large error in the MAE is $|e|$
(linear in the size of the error), whereas in the MSE it is $e^2$
(quadratic in the size of the error). Thus, very large errors have a
disproportionately large effect on the MSE compared to the MAE.*

### Exercise B2: Robust Regression (quantile trimming)

In quantile-trimmed robust regression, we modify our scoring function to
ignore a certain fraction of the largest and smallest residuals
(errors). For example, with a `trim` setting of `0.1`, we would ignore
the largest 10% and smallest 10% of the residuals when computing our
score. Quantile trimming can be implemented for both the MSE and the
MAE.

Please implement a quantile-trimmed version of the MSE scoring function
called `robust_mse`. This scoring function should have an extra keyword
argument `trim` which describes what fraction of the smallest errors to
ignore (and what fraction of the largest errors to ignore). Then, as in
the last section:

1.  Use your `fit_linear_model` function to fit a model using your
    robust MSE scoring function with a `trim` setting of `0.2`.
2.  Plot the resulting model predictions against the data, along with
    the MSE and MAE fits from earlier sections. Please make sure that
    all three prediction lines are shown on the same plot for easy
    comparison, and that they are distinguished using an
    appropriately-labeled legend.
3.  Comment on your findings.

In [ ]:
# TODO: Your code here

*[TODO: Your response here]*

### Exercise B3: Robust Regression (Margin loss)

Rather than say that we are going to ignore a certain fraction of the
residuals, we can instead define a “margin” $c$, where if
$\left|y_i - \hat{y}_i\right| < c$, we consider the prediction to be
“close enough” and assign it zero error. So, only errors larger than $c$
contribute to the loss. This leads to the following scoring function:

$$
\begin{aligned}
    \text{Margin Loss} = \frac{1}{n} \sum_{i=1}^n L_i \quad \text{where} \quad L_i = \begin{cases}
    0 & \text{if } \left|y_i - \hat{y}_i\right| < c \\
    \left|y_i - \hat{y}_i\right| & \text{otherwise}
    \end{cases}\;.
\end{aligned}
$$

As before, please implement this scoring function with an extra keyword
argument `margin` that specifies the value of $c$. Then:

1.  Use your `fit_linear_model` function to fit a model using your
    margin loss scoring function with a `margin` setting of `1`.
2.  Plot the resulting model predictions against the data, along with
    the MSE, MAE, and robust MSE fits from earlier sections. Please make
    sure that all four prediction lines are shown on the same plot for
    easy comparison, and that they are distinguished using an
    appropriately-labeled legend.
3.  Comment on your findings.

In [ ]:
# TODO: Your code here

*[TODO: Your response here]*

### Exercise B4: Regression with Unequal Cost of Errors

All the scoring functions we’ve seen so far treat overestimates and
underestimates equally. However, in some applications, one type of error
may be more costly than the other. For example, in predicting medical
diagnoses, underestimating the severity of a condition may have more
serious consequences than overestimating it.

To account for this, we can define a scoring function that assigns
different weights to overestimates and underestimates. For example, we
can define a *weighted MSE* as follows:

$$
\begin{aligned}
    \text{Weighted MSE} = \frac{1}{n} \sum_{i=1}^n \begin{cases}
    \alpha (y_i - \hat{y}_i)^2 & \text{if } y_i > \hat{y}_i \\
    \beta (y_i - \hat{y}_i)^2 & \text{otherwise}
    \end{cases}\;.
\end{aligned}
$$

Here, $\alpha$ plays the role of a cost of *underestimating* the true
value, while $\beta$ plays the role of a cost of *overestimating* the
true value.

As before, please implement this scoring function with extra keyword
arguments `underestimate_cost` and `overestimate_cost`. Then:

1.  Use your `fit_linear_model` function to fit a model using your
    weighted MSE scoring function with `underestimate_cost` set to `1`
    and `overestimate_cost` set to `2`. Then, try swapping the costs and
    otherwise adjusting these parameters. What happens when you allow
    one parameter to be much larger than the other?
2.  Plot the resulting model predictions against the data, comparing to
    previous model fits. Please make sure that all prediction lines are
    shown on the same plot for easy comparison, and that they are
    distinguished using an appropriately-labeled legend.
3.  Comment on your findings.

In [ ]:
# TODO: Your code here

*[TODO: Your response here]*

## Collaboration statement

In a markdown cell below, briefly list who or what you collaborated with
and how. Cite any sources here or with relevant inline comments in your
code. Acknowledge all contributors, both people and AI, and what
portions of this notebook they contributed. You do not need to cite or
acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant
assignment on [Gradescope](https://gradescope.com) via the “Upload
option” (guide
[here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)).
**Both files must be uploaded at the same time and the file names must
match the specification exactly for the autotesting to run
successfully.**

1.  `activity_score_functions.ipynb`: Your completed IPython notebook.
    You can obtain this via the “File→Download→Download .ipynb” menu
    option in Colab.
2.  `activity_score_functions.py`: Your completed IPython notebook as a
    Python file. You can obtain this via the “File→Download→Download
    .py” menu option in Colab. This file is used to provide line-level
    feedback on your submission.

You can submit multiple times, with only the most recent submission
(before the final due date) assessed for credit. Gradescope will run a
series of automated unit tests on your notebook (which may takes 10s of
seconds depending on the complexity of the notebook). Note that the
tests performed by Gradescope are limited. Passing all of the visible
tests does not guarantee that your submission correctly satisfies all of
the requirements of the assignment.